<a href="https://colab.research.google.com/github/Esaiasson/Machine_learning_WS_2025/blob/decision_tree_v2/A1/decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import time
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score,
    make_scorer
)

In [42]:
folder = ""
obesity_df_train_minmax_path = folder + "obesity_df_train_minmax_preprocessed.csv"
obesity_df_test_minmax_path = folder + "obesity_df_test_minmax_preprocessed.csv"
depression_df_train_minmax_path = folder + "depression_df_train_minmax_preprocessed.csv"
congressional_df_train_path = folder + "congressional_voting_preprocessed.csv"
rev_df_train_minmax_path = folder + "rev_df_lrn_minmax_preprocessed.csv"
rev_df_train_path = folder + "amazon_review_ID.shuf.lrn.csv"


In [43]:

obesity_df_train_minmax = pd.read_csv(obesity_df_train_minmax_path)
obesity_df_test_minmax = pd.read_csv(obesity_df_test_minmax_path)
#depression_df_train_minmax = pd.read_csv(depression_df_train_minmax_path)
#congressional_df_train = pd.read_csv(congressional_df_train_path)
#rev_df_train_minmax = pd.read_csv(rev_df_train_minmax_path)
#rev_df_train = pd.read_csv(rev_df_train_path)

#ev_df_train.head()

In [39]:
def train_deci_tree_with_grid(df, target_attribute, main_scorer, strategy):

  scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro'),
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      'min_samples_split': range(2,10,1)
  }

  if df.shape[0] <= 1000:
    param_grid["max_depth"] = range(5,15,1)
  elif df.shape[0] > 1000 and df.shape[0] <= 10000:
    param_grid["max_depth"] = range(10,20,1)
  else:
    param_grid["max_depth"] = range(10,50,1)

  x = df.loc[:, df.columns != target_attribute]
  y_raw = df[target_attribute]
  le = LabelEncoder()
  y = le.fit_transform(y_raw)

  start = time.perf_counter()

  tree = DecisionTreeClassifier(random_state=1)

  if strategy == "stratified":
    cv_strategy = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=1
    )
  else:
    cv_strategy = KFold(
      n_splits=5,
      shuffle=True,
      random_state=1
    )

  grid_search = GridSearchCV(
      estimator=tree,
      param_grid=param_grid,
      cv=cv_strategy,
      scoring=scoring,
      refit=main_scorer,
      verbose=True
  )

  grid_search.fit(x,y)

  results = pd.DataFrame(grid_search.cv_results_)
  elapsed = time.perf_counter() - start
  results["Completion_time"] = elapsed
  print("best accuracy", grid_search.best_score_)
  print(grid_search.best_estimator_)
  print("Time(s): ", elapsed)
  return results, le, grid_search.best_estimator_

In [25]:
def train_deci_tree_with_grid_holdout(df, target_attribute):
  pass


In [40]:
results_obesity_minmax, le_obesity, obesity_minmax_best_model = train_deci_tree_with_grid(obesity_df_train_minmax, "obesity_level_grouped", "accuracy", "stratified")
#results_depression_minmax, le_depression, depression_minmax_best_model = train_deci_tree_with_grid(depression_df_train_minmax, "depression", "recall", "stratified")
#results_congressional, le_congressional, congressional_best_model = train_deci_tree_with_grid(congressional_df_train, "class", "accuracy", "stratified")
#results_rev_minmax, le_rev, rev_minmax_best_model = train_deci_tree_with_grid(rev_df_train_minmax, "Class", "accuracy", "stratified")

Fitting 5 folds for each of 160 candidates, totalling 800 fits
best accuracy 0.7926535915579513
DecisionTreeClassifier(criterion='entropy', max_depth=16, min_samples_split=3,
                       random_state=1)
Time(s):  19.142168253999444


In [34]:
def get_top_results(results_df, main_scorer):
  mean_score_metrics = ["mean_test_f1", "mean_test_accuracy", "mean_test_precision", "mean_test_recall"]
  if f"mean_test_{main_scorer}" in mean_score_metrics:
    mean_score_metrics.remove(f"mean_test_{main_scorer}")
  mean_score_metrics.insert(0, f"mean_test_{main_scorer}")
  results_df["combined_rank"] = results_df["rank_test_accuracy"] + results_df["rank_test_precision"] + results_df["rank_test_recall"] + results_df["rank_test_f1"]
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  mean_score_metrics.append("combined_rank")
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant


In [35]:
processed_results_obesity_minmax = get_top_results(results_obesity_minmax, "accuracy")
#processed_results_depression_minmax = get_top_results(results_depression_minmax, "accuracy")
#processed_results_congressional = get_top_results(results_congressional, "accuracy")
#processed_results_rev_minmax = get_top_results(results_rev_minmax, "accuracy")

In [36]:
processed_results_obesity_minmax.head()

,mean_test_accuracy,mean_test_f1,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
49,0.792654,0.753407,0.753372,0.756644,5,entropy,16,3
73,0.792650,0.751682,0.752350,0.754234,14,entropy,19,3
59,0.792072,0.751458,0.750571,0.756754,15,entropy,17,5
57,0.792062,0.751411,0.751745,0.754545,16,entropy,17,3
65,0.791467,0.751007,0.750953,0.754419,23,entropy,18,3


In [130]:
processed_results_depression_minmax.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
54,0.825257,0.830777,0.826163,0.824631,30,entropy,7,8
55,0.825257,0.830777,0.826163,0.824631,30,entropy,7,9
48,0.825158,0.830687,0.826078,0.824523,38,entropy,7,2
49,0.825158,0.830687,0.826078,0.824523,38,entropy,7,3
50,0.825158,0.830687,0.826078,0.824523,38,entropy,7,4


In [128]:
processed_results_congressional.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
4,0.95753,0.958774,0.958199,0.958378,4,entropy,5,6
12,0.95753,0.958774,0.958199,0.958378,4,entropy,6,6
20,0.95753,0.958774,0.958199,0.958378,4,entropy,7,6
28,0.95753,0.958774,0.958199,0.958378,4,entropy,8,6
36,0.95753,0.958774,0.958199,0.958378,4,entropy,9,6


In [150]:
processed_results_rev_minmax.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
157,0.232167,0.269333,0.250731,0.261667,10,gini,14,7
158,0.231309,0.268000,0.250064,0.260667,16,gini,14,8
152,0.230167,0.266667,0.250731,0.260000,16,gini,14,2
154,0.229995,0.266667,0.250731,0.259667,18,gini,14,4
153,0.228928,0.265333,0.250064,0.258333,26,gini,14,3


In [1]:
results_obesity_minmax.head()

NameError: name 'results_obesity_minmax' is not defined

In [56]:
def pred_test_data(test_df,model, label_encoder,target_attribute):
  x_test = test_df.loc[:, test_df.columns != target_attribute]
  y_test_raw = test_df[target_attribute]
  y_test = label_encoder.transform(y_test_raw)

  start = time.perf_counter()

  y_pred = model.predict(x_test)

  elapsed = time.perf_counter() - start
  final_results = pd.DataFrame()
  final_results["accuracy"] = [accuracy_score(y_test, y_pred)]
  final_results["precision"] = [precision_score(y_test, y_pred, average="macro")]
  final_results["recall"] = [recall_score(y_test, y_pred, average="macro")]
  final_results["f1"] = [f1_score(y_test, y_pred, average="macro")]
  final_results["time"] = [elapsed]

  print("precision", precision_score(y_test, y_pred, average="macro"))
  print(final_results.head())


In [57]:
pred_test_data(obesity_df_test_minmax, obesity_minmax_best_model, le_obesity, "obesity_level_grouped")

precision 0.704465274840373
   accuracy  precision    recall        f1      time
0  0.737589   0.704465  0.719979  0.709198  0.003237
